# Generating Sparse Networks — Techniques

## Why Sparse Networks Matter

The human brain is a remarkable feat of efficiency. With approximately **86 billion neurons**, it consumes only about **20 watts** of power—roughly the energy used by a single light bulb. How does it achieve such efficiency while maintaining the computational capacity to think, learn, and remember?

One key answer: **sparsity**.

If every neuron connected to every other neuron, the number of synapses (connections) would be astronomical, and the brain's energy consumption would skyrocket. Instead, the brain uses **extreme sparsity**: in most brain regions, each neuron connects to only 1-10% of other neurons. This means the brain maintains ~100 trillion synapses rather than ~7 × 10¹⁸ if it were fully connected.

### The costs of connectivity

Each synapse is metabolically expensive:
- **Structural cost**: Maintaining the physical synaptic machinery
- **Signaling cost**: Operating neurotransmitter release and reuptake
- **Plasticity cost**: Updating synaptic weights during learning

By being sparse, biological networks achieve **redundancy** (multiple pathways for information) while **minimizing resource expenditure**.

### What makes a "good" sparse network?

A sparse network is not simply one with few edges—it must have *structure*. The previous notebooks introduced biological properties that distinguish biological networks from random ones:

- **Hierarchical**: Nodes are organized into levels, with information flowing from low to high (or vice versa)
- **Modular**: The graph decomposes into clusters with dense internal connections but sparse inter-module connections
- **Directional**: Edges have directionality reflecting information or resource flow
- **Redundant**: Multiple pathways exist between important nodes
- **Resource-efficient**: The structure reflects constraints (e.g., physical space, metabolic cost)
- **Weighted**: Different connections have different strengths, reflecting importance or capacity

In this notebook, we explore **graph construction techniques** and evaluate them against these properties.

## Properties of Biological Networks — A Recap

| Property | Biological Justification | Captured by ER | Captured by WS | Captured by BA | Captured by Spatial |
|----------|--------------------------|---|---|---|---|
| **Hierarchical** | Organization from sensory input to motor output | ✗ | ✗ | ✗ | ✗ |
| **Modular** | Specialized processing areas; efficient for learning | ✗ | ✓ (partial) | ✗ | ✓ (partial) |
| **Directional** | Synaptic transmission is unidirectional | ✗ | ✗ | ✗ | ✗ |
| **Redundant** | Robustness against neuron loss | ✓ (stochastic) | ✓ (by design) | ✓ (hub-dependent) | ✓ (stochastic) |
| **Resource-efficient** | Constraint by metabolic budget and physical space | ✗ | ✗ | ✗ | ✓ (distance-based) |
| **Weighted** | Synaptic strengths vary; reflect importance | ✗ | ✗ | ✗ | ✓ (can encode distance) |

Note: These are partial captures. We'll see that *no single method* reproduces all properties perfectly. That's why biological networks often use *combinations* of mechanisms.

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("Imports successful. Ready to generate sparse networks.")

## Classification of Graph Construction Methods

Graph construction methods can be classified along **four independent axes**:

### 1. Iterative vs. Non-iterative
- **Iterative**: The algorithm builds the graph step-by-step, where each step adds or modifies edges.
- **Non-iterative**: The algorithm generates all edges at once based on global rules.
- *Biological relevance*: Real brains evolve iteratively over development; non-iterative methods are faster computationally.

### 2. Online vs. Offline
- **Online**: The algorithm can incorporate new information (e.g., new nodes) during construction without recomputing everything.
- **Offline**: The algorithm requires all input information upfront; it's a one-shot computation.
- *Biological relevance*: Real neural systems develop online (new neurons are born, connections form as information arrives); this is crucial for learning and adaptation.

### 3. Adaptive vs. Non-adaptive
- **Adaptive**: The construction process responds to the network's own properties (e.g., preferential attachment based on current degree).
- **Non-adaptive**: The rules are fixed independent of the network state.
- *Biological relevance*: Biological networks are highly adaptive; synaptic plasticity allows the network to learn and reorganize.

### 4. Single-pass vs. Multi-pass
- **Single-pass**: The algorithm iterates through data/nodes only once.
- **Multi-pass**: The algorithm makes multiple passes, allowing corrections and refinements.
- *Biological relevance*: Learning often requires multiple passes (consolidation, replay); development involves multiple developmental waves.

---

## Quick Reference: Method Classification

| Model | Iterative | Online | Adaptive | Single-pass |
|-------|-----------|--------|----------|-------------|
| **Erdős-Rényi (ER)** | No | No | No | Yes |
| **Watts-Strogatz (WS)** | Yes | No | No | Yes |
| **Barabási-Albert (BA)** | Yes | Yes | Yes | Yes |
| **Spatial/Geometric** | No | No | No | Yes |

Notice: **Iterative, online, adaptive, multi-pass methods are biologically more plausible**, but computationally more expensive. Non-iterative offline methods are fast but often miss biological structure.

In [ ]:
def analyze_graph(G, name="Graph"):
    """Compute and print key graph metrics."""
    n = G.number_of_nodes()
    e = G.number_of_edges()
    max_edges = n * (n - 1) / 2
    density = e / max_edges if max_edges > 0 else 0
    cc = nx.average_clustering(G)
    
    # For disconnected graphs, compute largest component's path length
    if nx.is_connected(G):
        apl = nx.average_shortest_path_length(G)
    else:
        largest_cc = max(nx.connected_components(G), key=len)
        G_sub = G.subgraph(largest_cc).copy()
        if len(G_sub) > 1:
            apl = nx.average_shortest_path_length(G_sub)
        else:
            apl = 0
    
    print(f"\n--- {name} ---")
    print(f"Nodes: {n}, Edges: {e}")
    print(f"Density (1-sparsity): {density:.4f}")
    print(f"Sparsity: {1-density:.4f}")
    print(f"Avg Clustering Coefficient: {cc:.4f}")
    print(f"Avg Path Length (largest component): {apl:.4f}")
    
    return {
        "name": name,
        "n": n,
        "edges": e,
        "density": density,
        "sparsity": 1 - density,
        "cc": cc,
        "apl": apl
    }

print("Helper function defined: analyze_graph()")

## The Erdős-Rényi Model: Baseline for Random Sparsity

The **Erdős-Rényi (ER) graph** is the simplest random graph model. It generates a random graph by:
1. Creating $n$ nodes
2. For each pair of nodes, including the edge with probability $p$ (independently)

### Key Properties
- **Non-iterative, offline, non-adaptive, single-pass**: All edges are independent; the graph is "built all at once."
- **Poisson degree distribution**: Most nodes have roughly the same degree (around $p \cdot n$)
- **Low clustering**: ER graphs have very low clustering coefficients (~$p$) because edges are independent
- **Small-world property (partial)**: Short average path lengths despite random edges

### Sparsity and Connectivity
As $p$ decreases (graph becomes sparser), something dramatic happens: **the graph fragments**.

There's a critical threshold $p_c \approx 1/n$ below which the graph ceases to have a giant connected component. This is a **phase transition** in network science, analogous to percolation in physics.

### What ER *doesn't* capture
ER captures **none** of the biological properties:
- No hierarchy (all nodes are "equal" from the model's perspective)
- No modularity (edges are uniformly random)
- No directionality (undirected by default)
- No distance-dependence (a node can connect to anyone, anywhere)

**ER is a baseline**: useful for comparison but not biologically realistic.

In [ ]:
# Generate ER graphs at various sparsity levels
n = 100
p_values = [0.5, 0.2, 0.1, 0.05, 0.01]
er_results = []

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, p in enumerate(p_values):
    G = nx.erdos_renyi_graph(n, p, seed=42)
    result = analyze_graph(G, f"ER (p={p})")
    er_results.append(result)
    
    # Plot
    pos = nx.spring_layout(G, seed=42, iterations=20)
    ax = axes[idx]
    nx.draw_networkx_nodes(G, pos, node_color='skyblue', node_size=30, ax=ax)
    nx.draw_networkx_edges(G, pos, alpha=0.2, ax=ax)
    ax.set_title(f"ER (p={p})\nDensity: {result['density']:.3f}, CC: {result['cc']:.3f}")
    ax.axis('off')

# Remove extra subplot
axes[5].remove()

plt.tight_layout()
plt.savefig('/tmp/er_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("Summary: ER graphs become disconnected as p decreases")
print("="*60)

## The Watts-Strogatz Model: Adding Local Clustering

The **Watts-Strogatz (WS) model** addresses a key limitation of ER graphs: lack of clustering. It works in two steps:

### Algorithm
1. Start with a **ring lattice**: $n$ nodes arranged in a circle, each connected to the $k$ nearest neighbors
2. **Rewire** edges: For each edge, with probability $p$, disconnect it and reconnect to a random node

### Key Properties
- **Iterative, offline, non-adaptive, single-pass**: We build the lattice, then iteratively rewire
- **High clustering (when $p$ is small)**: Neighbors of a node tend to be neighbors of each other (because they came from the lattice)
- **Small-world**: Despite high clustering, average path length is low (due to rewired edges acting as "shortcuts")
- **Sparsity control**: By choosing $k \ll n$, we get sparse graphs from the start

### Biological Relevance
WS graphs capture **modularity** (local clustering creates modules) better than ER. However, they're still limited:
- The degree distribution is still approximately regular (not like biological networks with hubs)
- There's no spatial constraint (distance doesn't matter)
- No directionality

### Trade-off: Clustering vs. Path Length
- When $p = 0$: Pure lattice — high clustering, high path length
- When $p = 1$: Random rewiring — low clustering, low path length
- The "sweet spot" is intermediate $p$, where we get both high clustering AND short paths (small-world)

In [ ]:
# Generate WS graphs at various k and p values
n = 100
ws_results = []

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Parameters: (k, p) pairs
params = [(4, 0.0), (4, 0.3), (4, 1.0), (8, 0.0), (8, 0.3), (8, 1.0)]

for idx, (k, p) in enumerate(params):
    G = nx.watts_strogatz_graph(n, k, p, seed=42)
    result = analyze_graph(G, f"WS (k={k}, p={p})")
    ws_results.append(result)
    
    # Plot
    pos = nx.spring_layout(G, seed=42, iterations=20)
    ax = axes[idx]
    nx.draw_networkx_nodes(G, pos, node_color='lightcoral', node_size=30, ax=ax)
    nx.draw_networkx_edges(G, pos, alpha=0.2, ax=ax)
    ax.set_title(f"WS (k={k}, p={p})\nCC: {result['cc']:.3f}, APL: {result['apl']:.2f}")
    ax.axis('off')

plt.tight_layout()
plt.savefig('/tmp/ws_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("Key observation: p controls the clustering-vs-path-length trade-off")
print("Small p → high clustering but long paths")
print("Large p → low clustering but short paths")
print("="*60)

## The Barabási-Albert Model: Scale-free Sparsity

The **Barabási-Albert (BA) model** introduces a fundamentally different principle: **preferential attachment**. The idea is "rich get richer"—nodes that already have many connections are more likely to gain new connections.

### Algorithm
1. Start with $m_0$ nodes, all connected
2. Add nodes one at a time. Each new node forms $m$ edges to existing nodes
3. **Crucially**: The probability of connecting to an existing node is proportional to its current degree

### Key Properties
- **Iterative, online-like, adaptive, single-pass**: We add nodes one by one, and the network adapts based on current degrees
- **Power-law degree distribution**: Degree distribution follows $P(k) \propto k^{-\gamma}$ (usually $\gamma \approx 3$). This creates hubs—a few highly connected nodes and many sparsely connected nodes
- **Low clustering**: BA graphs actually have lower clustering than ER at the same density!
- **Sparse but robust**: Removing random nodes doesn't disconnect the graph, but removing hubs does

### Biological Relevance
BA captures **hub-like structure**, which does appear in some biological networks (e.g., hub neurons in C. elegans, hub regions in the mammalian brain). However:
- It doesn't naturally produce modularity
- It doesn't produce hierarchy
- Clustering is actually lower than in biological networks

### Why Power-law?
Power-law distributions are scale-free: they look similar at different scales. This is common in biological networks because it provides:
- **Robustness**: Removing random nodes has little effect
- **Efficiency**: Hubs can integrate information from many sources
- **But**: Vulnerability to targeted attacks on hubs

In [ ]:
# Generate BA graphs and compare degree distributions
n = 200
m_values = [1, 2, 3, 5]

fig, axes = plt.subplots(2, 4, figsize=(16, 10))
axes = axes.flatten()

ba_results = []

for idx, m in enumerate(m_values):
    # BA graph
    G_ba = nx.barabasi_albert_graph(n, m, seed=42)
    result = analyze_graph(G_ba, f"BA (m={m})")
    ba_results.append(result)
    
    # Plot graph
    pos = nx.spring_layout(G_ba, seed=42, iterations=20)
    ax = axes[idx]
    node_sizes = [G_ba.degree(node) * 10 for node in G_ba.nodes()]
    nx.draw_networkx_nodes(G_ba, pos, node_color='lightgreen', node_size=node_sizes, ax=ax, alpha=0.7)
    nx.draw_networkx_edges(G_ba, pos, alpha=0.1, ax=ax)
    ax.set_title(f"BA (m={m})\nSparsity: {result['sparsity']:.3f}")
    ax.axis('off')
    
    # Plot degree distribution
    degrees = [G_ba.degree(node) for node in G_ba.nodes()]
    ax = axes[idx + 4]
    ax.hist(degrees, bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
    ax.set_xlabel('Degree')
    ax.set_ylabel('Count')
    ax.set_title(f"Degree Distribution (m={m})")
    ax.set_yscale('log')
    ax.set_xscale('log')

plt.tight_layout()
plt.savefig('/tmp/ba_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("Key observation: BA produces power-law degree distributions")
print("This creates hubs, which are biologically relevant but...")
print("Clustering is low, and modularity is not captured.")
print("="*60)

## Spatial/Geometric Graphs: Distance-Dependent Connectivity

All the methods above treat space as irrelevant—a node can connect to any other node regardless of "distance." But real biological networks are embedded in physical space:

- Neurons occupy a 3D volume (the brain)
- Synaptic connections require physical proximity (axons and dendrites must meet)
- The cost of a connection increases with distance (longer axons require more materials and energy)

### Random Geometric Graphs (RGG)
The simplest spatial model:
1. Place $n$ nodes randomly in a unit square (or cube)
2. Add an edge between nodes $i$ and $j$ if their distance is below some threshold $r$

This captures **distance-dependent sparsity**. However, it doesn't weight edges by distance.

### Inverse-Square Weighted Geometric Graphs
A more biologically realistic variant:
1. Place $n$ nodes randomly in space
2. For each pair of nodes, compute probability of connection as $p_{ij} \propto 1/d_{ij}^2$ (or other power law)
3. Stochastically add edges based on this probability
4. Assign weights inversely related to distance (longer connections cost more)

### Why Inverse-Square?
The inverse-square law appears in physics (gravity, electromagnetism) and in neurobiology:
- Metabolic cost of maintaining an axon grows with length
- Signal attenuation over distance (though this is not purely inverse-square in biology)
- Space-filling principles: to maximize connectivity while minimizing total length, power-law distributions emerge

### Biological Relevance
Spatial graphs naturally capture:
- **Resource-efficiency**: Distance constraint limits total "wiring cost"
- **Modularity**: Nearby nodes cluster together, creating modules
- **Weighted structure**: Weights reflect distance-dependent costs

What they *still* miss:
- Hierarchy (spatial layout alone doesn't impose hierarchy)
- Directionality (edges are undirected)
- Adaptive plasticity (structure is fixed once generated)

In [ ]:
def generate_inverse_square_graph(n, threshold=0.1, power=2.0, seed=None):
    """
    Generate a spatial graph where connection probability is proportional to 1/d^power.
    
    Parameters:
    -----------
    n : int
        Number of nodes
    threshold : float
        Minimum probability below which edges are not created
    power : float
        Power law exponent (default 2.0 for inverse-square)
    seed : int or None
        Random seed for reproducibility
    
    Returns:
    --------
    G : networkx.Graph
        The generated spatial graph with node positions stored as 'pos' attribute
    """
    rng = np.random.default_rng(seed)
    
    # Generate random positions in [0, 1] x [0, 1]
    positions = {i: rng.random(2) for i in range(n)}
    
    G = nx.Graph()
    G.add_nodes_from(range(n))
    nx.set_node_attributes(G, positions, 'pos')
    
    # Add edges based on distance
    for i in range(n):
        for j in range(i+1, n):
            dist = np.linalg.norm(positions[i] - positions[j])
            if dist > 0:
                # Connection probability: inversely related to distance
                prob = min(1.0, 1.0 / (dist ** power))
                if prob > threshold and rng.random() < prob:
                    # Weight is inversely proportional to distance
                    weight = 1.0 / dist if dist > 0 else 1.0
                    G.add_edge(i, j, weight=weight, distance=dist)
    
    return G

# Generate spatial graphs with different parameters
n = 100
thresholds = [0.05, 0.10, 0.15]
spatial_results = []

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, threshold in enumerate(thresholds):
    G = generate_inverse_square_graph(n, threshold=threshold, power=2.0, seed=42)
    result = analyze_graph(G, f"Spatial (threshold={threshold})")
    spatial_results.append(result)
    
    # Get positions
    positions = nx.get_node_attributes(G, 'pos')
    
    ax = axes[idx]
    # Draw edges colored by weight
    for edge in G.edges():
        i, j = edge
        pos_i = positions[i]
        pos_j = positions[j]
        weight = G[i][j]['weight']
        ax.plot([pos_i[0], pos_j[0]], [pos_i[1], pos_j[1]], 
                color='gray', alpha=0.3, linewidth=0.5)
    
    # Draw nodes
    pos_array = np.array([positions[i] for i in range(n)])
    ax.scatter(pos_array[:, 0], pos_array[:, 1], 
              c='skyblue', s=50, edgecolors='darkblue', linewidth=0.5)
    
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_aspect('equal')
    ax.set_title(f"Spatial Graph (threshold={threshold})\nDensity: {result['density']:.3f}")
    ax.set_xlabel('X')
    ax.set_ylabel('Y')

plt.tight_layout()
plt.savefig('/tmp/spatial_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("Spatial graphs are biologically plausible:")
print("- Nodes cluster by proximity (creates modularity)")
print("- Distance constrains connectivity (resource-efficient)")
print("- But directionality is still missing")
print("="*60)

## Comprehensive Comparison: Methods vs. Biological Properties

Let's create a detailed comparison table evaluating each method against the biological properties we care about.

### Scoring System
- ✓ **Strong**: The method naturally captures this property
- ◐ **Partial**: The method captures the property to some degree
- ✗ **Weak/None**: The method does not capture this property

### Summary Table

| Method | Hierarchical | Modular | Directional | Redundant | Resource-eff | Weighted | Clustering | Sparsity | Remarks |
|--------|---|---|---|---|---|---|---|---|---|
| **ER** | ✗ | ✗ | ✗ | ◐ | ✗ | ✗ | ✗ | ✓ | Random baseline; no structure |
| **WS** | ✗ | ◐ | ✗ | ✓ | ✗ | ✗ | ✓ | ◐ | Captures small-world; good clustering |
| **BA** | ✗ | ✗ | ✗ | ◐ | ✗ | ✗ | ✗ | ◐ | Power-law hubs; vulnerable to targeted attack |
| **Spatial** | ✗ | ◐ | ✗ | ✓ | ✓ | ✓ | ◐ | ✓ | Distance-based; most biologically realistic so far |

### Key Insights

1. **No single method is perfect**: Each captures some biological properties but misses others.

2. **Spatial methods are closest**: They naturally incorporate:
   - Distance-dependent connectivity (resource efficiency)
   - Weighted edges (reflecting cost/importance)
   - Emergent modularity (nearby nodes cluster)

3. **Still missing**:
   - **Hierarchy**: None of these methods impose top-down organization
   - **Directionality**: These are all undirected (though easy to extend)
   - **Adaptive plasticity**: Once generated, the structure is fixed

4. **Hybrid approaches**: Real biological networks likely use **combinations** of mechanisms:
   - Spatial embedding provides the "scaffold"
   - Preferential attachment refines the structure (like learning)
   - Pruning (removing weak connections) shapes the final network
   - Hierarchy emerges from functional organization, not from the generative mechanism

### The Missing Ingredient: Pruning

A key observation: Biological development often follows **over-build then prune**:
1. Start with a dense spatial scaffold
2. Refine through activity-dependent pruning (remove weak connections)
3. Result: sparse, structured network

This is the subject of **Notebook 5**. For now, spatial generation is a good foundation.

In [ ]:
# Create a comprehensive visualization comparing all methods at matched sparsity

# Generate graphs with roughly matched sparsity (density ~0.05)
n = 100
graphs = {
    'ER (p=0.05)': nx.erdos_renyi_graph(n, 0.05, seed=42),
    'WS (k=4, p=0.5)': nx.watts_strogatz_graph(n, 4, 0.5, seed=42),
    'BA (m=3)': nx.barabasi_albert_graph(n, 3, seed=42),
    'Spatial (th=0.1)': generate_inverse_square_graph(n, threshold=0.1, seed=42),
}

# Analyze and collect results
comparison_results = {}
fig, axes = plt.subplots(2, 4, figsize=(16, 10))
axes = axes.flatten()

for idx, (name, G) in enumerate(graphs.items()):
    result = analyze_graph(G, name)
    comparison_results[name] = result
    
    # Visualization: spring layout for non-spatial, spatial coordinates for spatial
    if 'Spatial' in name:
        positions = nx.get_node_attributes(G, 'pos')
        pos_array = np.array([positions[i] for i in range(n)])
        ax = axes[idx]
        for edge in G.edges():
            i, j = edge
            ax.plot([positions[i][0], positions[j][0]], 
                   [positions[i][1], positions[j][1]], 
                   color='gray', alpha=0.2, linewidth=0.5)
        ax.scatter(pos_array[:, 0], pos_array[:, 1], 
                  c='skyblue', s=30, edgecolors='darkblue', linewidth=0.5)
        ax.set_aspect('equal')
    else:
        pos = nx.spring_layout(G, seed=42, iterations=20)
        ax = axes[idx]
        nx.draw_networkx_nodes(G, pos, node_color='skyblue', node_size=30, ax=ax)
        nx.draw_networkx_edges(G, pos, alpha=0.2, ax=ax)
    
    ax.set_title(f"{name}\nDensity: {result['density']:.3f}, CC: {result['cc']:.3f}")
    ax.axis('off')

# Metrics comparison in the remaining subplots
metrics = ['density', 'cc', 'apl']
metric_labels = ['Density', 'Clustering Coeff', 'Avg Path Length']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for metric_idx, (metric, label) in enumerate(zip(metrics, metric_labels)):
    ax = axes[4 + metric_idx]
    names = list(comparison_results.keys())
    values = [comparison_results[name][metric] for name in names]
    
    bars = ax.bar(range(len(names)), values, color=colors)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right')
    ax.set_ylabel(label)
    ax.set_title(f"{label} Comparison")
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/comprehensive_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON AT MATCHED SPARSITY")
print("="*80)
for name, result in comparison_results.items():
    print(f"\n{name}:")
    print(f"  Density: {result['density']:.4f}")
    print(f"  Clustering: {result['cc']:.4f}")
    print(f"  Avg Path Length: {result['apl']:.4f}")
print("="*80)

## Key Takeaways

### What We've Learned

1. **Sparsity is fundamental**: Biological networks are sparse not by accident, but by design. Sparsity saves energy while maintaining functionality.

2. **Different generation methods capture different properties**:
   - **Erdős-Rényi**: Simplest random model; captures none of the biological properties. Useful as a baseline.
   - **Watts-Strogatz**: Captures clustering and small-world properties; good for understanding modularity but lacks distance-dependence.
   - **Barabási-Albert**: Produces power-law hubs, reflecting some biological networks; but lacks modularity and clustering.
   - **Spatial/Geometric**: Most biologically plausible so far; captures distance-dependence, emergent modularity, and resource-efficiency.

3. **No single method is complete**: Each method is biologically inspired but falls short in different ways. Real networks likely combine mechanisms:
   - Spatial embedding for resource efficiency
   - Adaptive refinement (learning) to strengthen important connections
   - Pruning to remove weak or unnecessary connections
   - Hierarchical organization imposed by functional structure

4. **Sparsity and clustering are linked**: To achieve biological sparsity while maintaining robustness, networks need clustering (modular structure). Random ER graphs sacrifice clustering to be sparse.

5. **Distance matters**: Spatial constraints are perhaps the most fundamental biological constraint. Incorporating space explains several emergent properties: modularity, weighted structure, and realistic connectivity costs.

### The Road Ahead

- **Notebook 4**: Hybrid methods combining multiple mechanisms (spatial + preferential attachment, etc.)
- **Notebook 5**: Pruning and refinement—how to sculpt an initial dense network into a sparse, structured one
- **Notebook 6**: Adding directionality and hierarchy
- **Notebook 7**: Learning and adaptation—how networks evolve over time

### Recommended Next Steps

For your own experiments:
1. Experiment with different sparsity levels; observe phase transitions
2. Try generating graphs at fixed sizes with varying parameters; build intuition
3. Compare real neural data (if available) against these models
4. Combine methods: start with spatial, then apply preferential attachment, then prune

Remember: **The best model depends on your goals**. For understanding metabolic constraints, spatial methods shine. For understanding robustness and information integration, scale-free properties matter. For understanding learning and flexibility, adaptive mechanisms are essential.

In [ ]:
# Final summary: Create a table of all results
import pandas as pd

# Compile all results
all_results = []
all_results.extend(er_results)
all_results.extend(ws_results)
all_results.extend(ba_results)
all_results.extend(spatial_results)

# Create DataFrame
df = pd.DataFrame(all_results)
df = df[['name', 'n', 'edges', 'density', 'sparsity', 'cc', 'apl']]
df.columns = ['Method', 'Nodes', 'Edges', 'Density', 'Sparsity', 'Clustering', 'Avg Path Length']

print("\n" + "="*100)
print("SUMMARY TABLE: All Generated Networks")
print("="*100)
print(df.to_string(index=False))
print("="*100)

print("\nNotebook complete! You've explored the landscape of sparse network generation methods.")
print("Next: Notebook 4 will introduce hybrid methods combining spatial structure with adaptive mechanisms.")